# Reconstruct pool state block-by-block

Turn the irregular swap events into one aligned, gap-free per-pool series
(price + active liquidity), trim to the study window, and save one CSV per pool
under `S.processed_dir`. Parameters come from `arblib.config.STUDY`.

In [1]:
%load_ext autoreload
%autoreload 2

from arblib import data_io, preprocessing as pp
from arblib import formulas as f
from arblib import modeling, regime
from arblib.config import STUDY as S, LIQUIDITY_FILES

REGIME = S.active_regime             # same window as extract (arblib.config)
if S.test_mode:
    win = regime.custom_window(S, S.test_start, S.test_end)
    print(f"[TEST MODE] processing custom window | study starts {win['study_start']} "
          f"(= test_start + {S.test_lead_min} min warm-up)")
else:
    win = regime.study_window(S, REGIME)
    print(f"processing regime: {REGIME} | study starts {win['study_start']}")
STUDY_START = win["study_start"]

processing regime: mid | study starts 2026-04-07 00:00:00


## 0. Load the raw swap extracts

In [2]:
dfs = data_io.load_pool_csvs(S.swaps_dir)

Loaded: df_uniswap_swap.csv
        amount0                amount1         dex  evt_block_number  \
0   -3841311285    1791983531519317948  uniswap_v3          24822884   
1 -233726488341  109042974122184214283  uniswap_v3          24822884   
2 -196982210504   91960274128190799872  uniswap_v3          24822886   
3     262217106    -122425795333296691  uniswap_v3          24822886   
4    1369281053    -639114432732468996  uniswap_v3          24822887   

                evt_block_time  fee             liquidity  nb_swaps  \
0  2026-04-06 20:00:11.000 UTC  100    186713476356542916         1   
1  2026-04-06 20:00:11.000 UTC  500  14250197234964559077         1   
2  2026-04-06 20:00:35.000 UTC  500  14250454729234917041         1   
3  2026-04-06 20:00:35.000 UTC  100    186713476356542916         3   
4  2026-04-06 20:00:47.000 UTC  100    186713476356542916         7   

                                         pool  \
0  0xe0554a476a092703abdb3ef35c80e0d76d32939f   
1  0x88e6a0c2d

## 1. End-of-block row per pool & block (already done in SQL)

The aggregated swap queries already return one end-of-block row per `(pool, block)` with
`nb_swaps` and `gas_price_max` / `gas_price_med`, so the old `count_swaps` + `clean_all`
(keep-latest-per-block) step is now a pass-through.

In [3]:
# nb_swaps and the single end-of-block row per (pool, block) are already produced by the
# aggregated Dune swap queries, so count_swaps / keep_latest are no longer needed here.
filtered_dfs = dfs

## 2. Split each DEX into one series per pool

In [4]:
pool_dfs = pp.split_by_pool(filtered_dfs)

Created uniswap_1: 33815 rows
Created uniswap_2: 20093 rows
Created uniswap_3: 4019 rows
Created uniswap_4: 4241 rows
Created uniswap_5: 2516 rows
Created uniswap_6: 194 rows
Created uniswap_7: 106 rows
Created uniswap_8: 5 rows
Created pancake_1: 3574 rows
Created pancake_2: 13423 rows
Created pancake_3: 26 rows
Created pancake_4: 9 rows

Total: 12 dataframes
['uniswap_1', 'uniswap_2', 'uniswap_3', 'uniswap_4', 'uniswap_5', 'uniswap_6', 'uniswap_7', 'uniswap_8', 'pancake_1', 'pancake_2', 'pancake_3', 'pancake_4']


## 3. Drop pools that trade too rarely to reconstruct

In [5]:
pool_dfs, dropped_pools = pp.filter_pools_by_swap_gap(pool_dfs, S.max_gap_blocks)

k = 6000 blocks
Kept 10 pools

Kept pools:
  uniswap_1: 33815 swaps, max consecutive gap = 12 blocks
  uniswap_2: 20093 swaps, max consecutive gap = 48 blocks
  uniswap_3: 4019 swaps, max consecutive gap = 149 blocks
  uniswap_4: 4241 swaps, max consecutive gap = 111 blocks
  uniswap_5: 2516 swaps, max consecutive gap = 388 blocks
  uniswap_6: 194 swaps, max consecutive gap = 4145 blocks
  uniswap_7: 106 swaps, max consecutive gap = 2357 blocks
  pancake_1: 3574 swaps, max consecutive gap = 169 blocks
  pancake_2: 13423 swaps, max consecutive gap = 58 blocks
  pancake_3: 26 swaps, max consecutive gap = 4533 blocks

Dropped 2 pools:
  uniswap_8: max gap = 15508 blocks, tail gap = 966 blocks
  pancake_4: max gap = 10888 blocks, tail gap = 3951 blocks


## 3b. Drop dynamic-fee pools

The execution-price math needs a single fixed fee per pool, so pools whose `fee`
varies (e.g. Uniswap v4 dynamic-fee hooks) are excluded before saving.

In [6]:
pool_dfs, dropped_fee_pools = pp.filter_pools_by_constant_fee(pool_dfs)

Kept 10 constant-fee pools


## 4. Reconstruct a dense, forward-filled series per pool

In [7]:
reconstructed_pools, global_min, global_max, block_time_map = pp.reconstruct_pool_timeseries(pool_dfs)

Global block range: 24822884 to 24867119
Total blocks: 44236

uniswap_1: 33815 trades -> 44236 blocks (100.0%)
uniswap_2: 20093 trades -> 44236 blocks (100.0%)
uniswap_3: 4019 trades -> 44229 blocks (100.0%)
uniswap_4: 4241 trades -> 44219 blocks (100.0%)
uniswap_5: 2516 trades -> 44154 blocks (99.8%)
uniswap_6: 194 trades -> 43376 blocks (98.1%)
uniswap_7: 106 trades -> 43281 blocks (97.8%)
pancake_1: 3574 trades -> 44236 blocks (100.0%)
pancake_2: 13423 trades -> 44236 blocks (100.0%)
pancake_3: 26 trades -> 43626 blocks (98.6%)

Created 10 reconstructed time series


## 4b. Reconstruct active liquidity per block

Rebuild each pool's `liquidity` into the running active-liquidity state using the
mint/burn events (in-range deltas applied between swaps, held constant otherwise).

In [8]:
liq_dfs = data_io.load_pool_csvs(S.liquidity_dir, files=LIQUIDITY_FILES)
reconstructed_pools = pp.reconstruct_liquidity_states(reconstructed_pools, liq_dfs)

Loaded: df_uniswap_liq.csv
          dex  evt_block_number               evt_block_time  evt_index  \
0  uniswap_v3          24822893  2026-04-06 20:01:59.000 UTC        382   
1  uniswap_v3          24822924  2026-04-06 20:08:11.000 UTC        492   
2  uniswap_v3          24822948  2026-04-06 20:12:59.000 UTC        209   
3  uniswap_v3          24822961  2026-04-06 20:15:35.000 UTC        381   
4  uniswap_v3          24822968  2026-04-06 20:16:59.000 UTC        200   

   liquidity_delta                                        pool  tick_lower  \
0  648256183595339  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8      184200   
1    2775571932701  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640      199630   
2    3460728726192  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640      198340   
3 -258437513512663  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640      198950   
4  -65600461257188  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640      198550   

   tick_upper  
0      203220  
1      206550  
2    

## 4c. Drop pools with invalid (null / zero) active liquidity

A pool with no in-range liquidity (`L = 0`) or undefined `L` (before its first swap) has no
executable price — the round-trip math divides by `L`, and `log(L)` is `-inf` at 0, which poisons
the liquidity-growth covariate downstream. Pools with more than `S.max_invalid_liquidity_frac` of
blocks null or `<= 0` are dropped here, before the MEV proxies and save.

In [9]:
reconstructed_pools, dropped_liq_pools = pp.filter_pools_by_liquidity(
    reconstructed_pools, S.max_invalid_liquidity_frac)

Kept 10 pools with <= 10% invalid (null / zero) liquidity


## 5a. MEV friction proxies

Two decay-weighted MEV series per pool: `mev_intensity` (recent top priority tip,
`gas_price_max - base_fee`) and `contest_frequency` (recent rate of same-block races,
`nb_swaps >= 2`), each decayed over past swap blocks with horizon `S.mev_horizon_blocks`.
No forward-fill — a quiet block inherits no stale competition value (gets `NaN`).

In [10]:
chain_gas = data_io.load_chain_gas(S.gas_path)
reconstructed_pools = f.add_mev_intensity(reconstructed_pools, chain_gas, S.mev_horizon_blocks)
reconstructed_pools = f.add_contest_freq(reconstructed_pools, S.mev_horizon_blocks)

uniswap_1: mev_intensity max 9.118e+10 wei
uniswap_2: mev_intensity max 3.363e+11 wei
uniswap_3: mev_intensity max 3.265e+11 wei
uniswap_4: mev_intensity max 7.626e+09 wei
uniswap_5: mev_intensity max 7.034e+10 wei
uniswap_6: mev_intensity max 2.130e+10 wei
uniswap_7: mev_intensity max 1.342e+08 wei
pancake_1: mev_intensity max 2.194e+10 wei
pancake_2: mev_intensity max 2.194e+10 wei
pancake_3: mev_intensity max 1.290e+08 wei
uniswap_1: nb_swaps_ewma max 9.43
uniswap_2: nb_swaps_ewma max 6.45
uniswap_3: nb_swaps_ewma max 0.86
uniswap_4: nb_swaps_ewma max 1.25
uniswap_5: nb_swaps_ewma max 0.83
uniswap_6: nb_swaps_ewma max 0.35
uniswap_7: nb_swaps_ewma max 0.18
pancake_1: nb_swaps_ewma max 1.17
pancake_2: nb_swaps_ewma max 2.05
pancake_3: nb_swaps_ewma max 0.32


## 5. Trim to the study window

In [11]:
filtered_pools = pp.filter_by_start_time(reconstructed_pools, STUDY_START)

uniswap_1: 44236 -> 43042 rows
uniswap_2: 44236 -> 43042 rows
uniswap_3: 44236 -> 43042 rows
uniswap_4: 44236 -> 43042 rows
uniswap_5: 44236 -> 43042 rows
uniswap_6: 44236 -> 43042 rows
uniswap_7: 44236 -> 43042 rows
pancake_1: 44236 -> 43042 rows
pancake_2: 44236 -> 43042 rows
pancake_3: 44236 -> 43042 rows

Filtered all pools by time >= 2026-04-07 00:00:00


## 6. Save one CSV per pool

In [12]:
data_io.save_processed_pools(filtered_pools, S.processed_dir)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/uniswap_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/uniswap_2.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/uniswap_3.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/uniswap_4.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/uniswap_5.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/uniswap_6.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/uniswap_7.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/data_analysis/processed/pancake_1.csv
Saved: /Users/ma

## 7. Save the common (pool-independent) modeling covariates

Persist the covariates every pool pair shares to `S.common_covariates_dir`
(`modeling/covariates/common_covariates/`):

- **`CEX_volatility.parquet`** — `[time, ewma_vol]`: RiskMetrics EWMA volatility of the
  `token0/token1` exchange rate `R = P_X/USD / P_Y/USD` (log returns differenced over time,
  `var_t = λ·var_{t-1} + (1-λ)·r_t²`, `λ = exp(-1/S.vol_horizon_min)`). Minute grid — joined to
  blocks by a backward merge on `time` downstream.
- **`chain_covariates.parquet`** — `[block_number, time, log_base_fee_per_gas, gas_util,
  log1p_tip_p50, log1p_tip_p90]`: base fee in logs, block fullness `gas_used/gas_limit`, and the
  per-block priority-tip p50/p90 as `log(1+tip)`. Joined by exact `block_number`.

Also creates (empty) `pool_pair_dependant_covariates/` for the per-pool-pair features built
later. Both files span the full extract window (warm-up included); downstream joins select the
study blocks.

In [13]:
x_usd = data_io.load_price_series(S.x_price_path)
y_usd = data_io.load_price_series(S.y_price_path)

modeling.save_common_covariates(x_usd, y_usd, chain_gas, S.common_covariates_dir, S.vol_horizon_min)
S.pair_covariates_dir.mkdir(parents=True, exist_ok=True)

Saved common covariates (CEX_volatility, chain_covariates) to /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/mid_vol/modeling/covariates/common_covariates
